In [98]:
import importlib
from lago.lago import LinkStream, lago_communities


## Declare time links according to the following format:
# <source node>, <target node>, <time instant>
## Values must be integers

time_links = [
    [2, 3, 0],
    [0, 1, 2],
    [2, 3, 3],
    [3, 4, 5],
    [2, 3, 6],
    [2, 4, 7],
    [0, 1, 8],
    [1, 2, 9],
    [3, 4, 9],
    [0, 2, 10],
    [1, 2, 11],
    [3, 4, 13],
    [1, 2, 14],
    [2, 4, 16],
    [0, 1, 17],
    [0, 1, 18],
    [2, 3, 18],
    [3, 4, 19],
]

## Initiate empty temporal network (as a link stream)
my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

# NOTE time links can also be imported from txt files with the read_txt() method

## Display linkstream informations
print(f"The link stream consists of {my_linkstream.nb_edges} temporal edges (or time links) accross {my_linkstream.nb_nodes} nodes and {my_linkstream.network_duration} time steps, of which only {my_linkstream.nb_timesteps} contain activity.")

## Compute dynamic communities
dynamic_communities = lago_communities(
    my_linkstream,
    nb_iter=3, # run LAGO 3 times and return best result
    )

# Each dynamic community is represented by a list of (<node>, <time instant>)

print(f"{len(dynamic_communities)} dynamic communities have been found")
print(dynamic_communities)
import lago.l_modularity_function
importlib.reload(lago.l_modularity_function)
from lago.l_modularity_function import longitudinal_modularity

## Compute Longitudinal Modularity score
## (the higher the better / maximum is 1)
long_mod_score = longitudinal_modularity(
    my_linkstream, 
    dynamic_communities,
    lex_type="CM"
    )

print(f"Dynamic communities detected on the linkstream have a Longitudinal Modularity score of {long_mod_score} ")

The link stream consists of 18 temporal edges (or time links) accross 5 nodes and 20 time steps, of which only 16 contain activity.
2 dynamic communities have been found
{0: {(0, 2), (0, 5), (1, 6), (0, 8), (0, 14), (1, 3), (1, 9), (0, 11), (0, 17), (2, 14), (1, 12), (1, 18), (2, 11), (1, 15), (0, 7), (1, 2), (0, 4), (0, 10), (0, 16), (1, 5), (1, 11), (0, 13), (2, 10), (1, 8), (1, 14), (2, 13), (1, 17), (0, 3), (0, 9), (1, 4), (0, 6), (0, 12), (2, 9), (1, 7), (1, 13), (0, 15), (2, 12), (1, 10), (1, 16), (0, 18)}, 1: {(3, 4), (3, 1), (3, 7), (4, 9), (4, 6), (4, 12), (3, 10), (3, 16), (4, 15), (3, 13), (3, 19), (4, 18), (2, 2), (2, 5), (2, 17), (3, 0), (4, 5), (3, 3), (3, 9), (4, 8), (4, 14), (3, 6), (3, 12), (3, 18), (4, 11), (4, 17), (3, 15), (2, 4), (2, 1), (2, 7), (2, 16), (3, 2), (4, 7), (3, 5), (3, 11), (4, 10), (3, 8), (3, 14), (4, 16), (4, 13), (4, 19), (3, 17), (2, 0), (2, 3), (2, 6), (2, 18)}}
source:0, target:0, c:0 coexistence:17
source:0, target:1, c:0 coexistence:17
source:

In [81]:
import torch
from collections import defaultdict

def compute_soft_Tu(P, K):
    """
    |T_{u∈C}| = Σ_t p(u,t,C)
    P[u]: dict[t -> 1D tensor(K)]
    返回: Tu[u] -> 1D tensor(K)
    """
    Tu = {}
    device = next(iter(next(iter(P.values())).values())).device
    for u, td in P.items():
        acc = torch.zeros(K, device=device)
        for t, pvec in td.items():
            acc = acc + pvec  # 不用 in-place +=，更安全
        Tu[u] = acc
    return Tu


def compute_soft_Tuv(P, K):
    """
    |T_{uv∈C}| = Σ_t p(u,t,C) * p(v,t,C)
    返回: Tuv[(u,v)] -> 1D tensor(K)
    """
    Tuv = {}
    device = next(iter(next(iter(P.values())).values())).device
    nodes = list(P.keys())
    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            t_common = set(P[u].keys()) & set(P[v].keys())
            acc = torch.zeros(K, device=device)
            for t in t_common:
                acc = acc + P[u][t] * P[v][t]
            Tuv[(u, v)] = acc
    return Tuv


def compute_strength_C(P, K):
    """
    Strength_C(t) = 1 - ∏_u (1 - p(u,t,C))
    |T_C| = Σ_t Strength_C(t)
    返回:
        strength: dict[c][t] = strength_C(t)（float）
        T_C_len: 1D tensor(K) 软版 |T_C|
    """
    all_times = sorted({t for u in P for t in P[u].keys()})
    device = next(iter(next(iter(P.values())).values())).device

    strength = {c: {} for c in range(K)}

    for t in all_times:
        prod_term = torch.ones(K, device=device)
        for u, td in P.items():
            if t in td:
                prod_term = prod_term * (1 - td[t])
        strength_t = 1 - prod_term
        for c in range(K):
            strength[c][t] = float(strength_t[c].detach().cpu())

    T_C_len = torch.tensor(
        [sum(strength[c][t] for t in all_times) for c in range(K)],
        dtype=torch.float32,
        device=device,
    )
    return strength, T_C_len


def compute_soft_csc(P):
    """
    soft CSC:
    η_u = Σ_t 0.5 * ||p(u,t) - p(u,t_prev)||_1
    one-hot 时严格等于“社区切换次数”
    返回: eta_total (标量 tensor)
    """
    device = next(iter(next(iter(P.values())).values())).device
    eta_total = torch.zeros(1, device=device)

    for u, td in P.items():
        times = sorted(td.keys())
        for i in range(1, len(times)):
            diff = torch.abs(td[times[i]] - td[times[i-1]]).sum() * 0.5
            eta_total = eta_total + diff

    return eta_total.squeeze(0)


def soft_l_modularity_loss(edges, P, omega=2.0, lex_type="EMM"):
    """
    对齐论文 Eq.(10)(11)(12)(16) + 作者解释的“2**(u!=v)”风格的 soft 版 L-Modularity。

    参数：
        edges: list[(u, v, t)]，无向边（link stream）
        P: dict[u][t] = 1D tensor(K)，为 TGN 输出的 soft 社区分布
        omega: 平滑项权重（对应论文的 ω）
        lex_type: "ECM" / "EJM" / "EMM"

    返回：
        loss = -Q_L*   （最小化 loss 等价于最大化 L-Modularity）
        Q = Q_L* 本身（标量 tensor）
    """
    # 取设备 & 维度
    any_u = next(iter(P))
    any_t = next(iter(P[any_u]))
    K = P[any_u][any_t].shape[0]
    device = P[any_u][any_t].device

    nodes = sorted(P.keys())
    all_times = sorted({t for (_, _, t) in edges})
    T_len = float(max(all_times) -min(all_times)+1)
    m = float(len(edges))

    # 度数 k_u
    ku = defaultdict(float)
    for u, v, t in edges:
        ku[u] += 1.0
        ku[v] += 1.0

    # 观测项：L_{uv∈C}^soft = Σ_t A_uv(t) p(u,t,C)p(v,t,C)
    LuvC = defaultdict(lambda: torch.zeros(K, device=device))
    for u, v, t in edges:
        a, b = (u, v) if u <= v else (v, u)
        pu = P[u][t]  # shape [K]
        pv = P[v][t]
        LuvC[(a, b)] = LuvC[(a, b)] + pu * pv

    # 时间相关量：|T_u∈C|, |T_uv∈C|, |T_C|
    Tu = compute_soft_Tu(P, K)
    Tuv = compute_soft_Tuv(P, K)
    strength, T_C_len = compute_strength_C(P, K)

    # 节点是否曾属于社区 C（用于 EJM 的 C∈C_u∩C_v）
    membership = {
        u: (Tu[u] > 0) for u in nodes  # bool tensor(K)
    }

    # 平滑项 η_u
    eta_total = compute_soft_csc(P)

    # 主和项: Σ_C Σ_{u,v} [L_{uv∈C} - E^*_{uv∈C}]
    Q_sum = torch.zeros(1, device=device)
    print(Tu)
    print(ku)
    print(T_len)
    print(T_C_len)
    for c in range(K):
        for i, u in enumerate(nodes):
            for j, v in enumerate(nodes[i:]):
                v = nodes[i + j]
                a, b = u, v
                L_vec = LuvC.get((a, b), torch.zeros(K, device=device))

            
                L = L_vec[c]  # scalar tensor

                # 期望 E^*[L_{uv∈C}]，对应 Eq.(10)(11)(12)
                if lex_type == "ECM":
                    # ECM: k_u k_v / (2m) * |T_uv∈C|/|T|
                    Tuv_c = Tuv.get((a, b), torch.zeros(K, device=device))[c]
                    E = (ku[a] * ku[b]) / (2.0 * m) * (Tuv_c / T_len)

                elif lex_type == "EJM":
                    # EJM: k_u k_v / (2m) * |T_C|/|T| if C∈C_u ∩ C_v else 0
                    if membership[a][c] and membership[b][c]:
                        #E = (ku[a] * ku[b]) / (2.0 * m) * (T_C_len[c] / T_len)
                        E = 2 ** (u != v) * (ku[a] * ku[b]) * (T_C_len[c] / T_len) / (2.0 * m) ** 2
                    else:
                        E = torch.zeros((), device=device)

                elif lex_type == "EMM":
                    # EMM: k_u k_v / (2m) * sqrt(|T_u∈C||T_v∈C|)/|T|
                    Tu_c = Tu[a][c]
                    Tv_c = Tu[b][c]
                    E = (ku[a] * ku[b]) / (2.0 * m) * (
                        torch.sqrt(Tu_c * Tv_c + 1e-12) / T_len
                    )
                else:
                    raise ValueError("lex_type must be 'ECM', 'EJM', or 'EMM'")

                # 对应论文 Σ_{u,v}，我们只遍历 a<=b，所以要补 uv+vu
                # factor = 2.0 if a != b else 1.0
        print(f"expectation:{E}")
        Q_sum = Q_sum + L / (2 * m) - E


    print(f"Modularity term: {Q_sum}")
    smooth_penalty = omega * eta_total / (2.0 * m)
    Q = Q_sum - smooth_penalty

    # 训练时最小化 loss = -Q
    loss = -Q
    return loss, Q


import torch
# =====================
# 2. 你的测试数据
# =====================

time_links = [
    [2, 3, 0],
    [0, 1, 2],
    [2, 3, 3],
    [3, 4, 5],
    [2, 3, 6],
    [2, 4, 7],
    [0, 1, 8],
    [1, 2, 9],
    [3, 4, 9],
    [0, 2, 10],
    [1, 2, 11],
    [3, 4, 13],
    [1, 2, 14],
    [2, 4, 16],
    [0, 1, 17],
    [0, 1, 18],
    [2, 3, 18],
    [3, 4, 19],
]

# 社区划分: C -> {(u,t)}
comm = {0: {(3, 4), (4, 9), (3, 1), (3, 7), (4, 6), (4, 12), (3, 10), (3, 16), (4, 15), (3, 13), (3, 19), (4, 18), (2, 2), (2, 5), (2, 17), (3, 0), (4, 5), (3, 3), (3, 9), (4, 8), (4, 14), (3, 6), (3, 12), (4, 11), (4, 17), (3, 18), (3, 15), (2, 4), (2, 1), (2, 7), (2, 16), (3, 2), (4, 7), (3, 5), (3, 11), (4, 10), (4, 16), (3, 8), (3, 14), (4, 13), (4, 19), (3, 17), (2, 0), (2, 3), (2, 6), (2, 18)}, 1: {(0, 2), (0, 5), (1, 6), (0, 8), (0, 14), (1, 3), (1, 9), (0, 11), (0, 17), (2, 14), (1, 12), (1, 18), (2, 11), (1, 15), (0, 7), (1, 2), (0, 4), (0, 10), (0, 16), (1, 5), (1, 11), (0, 13), (2, 10), (1, 8), (1, 14), (2, 13), (1, 17), (0, 3), (0, 9), (1, 4), (0, 6), (0, 12), (2, 9), (1, 7), (1, 13), (0, 15), (2, 12), (1, 10), (1, 16), (0, 18)}}
# =====================
# 3. 把硬分配转换成严格 one-hot 的 soft 输入 P
# =====================

K = 5  # 你要求的社区向量维度

# 所有节点
nodes = set()
for u, v, t in time_links:
    nodes.add(u)
    nodes.add(v)

# (u,t) -> 所属社区 id（0 或 1）；如果不在任何社区中，就认为不属于任何社区
membership = {}
for c in comm:
    for (u, t) in comm[c]:
        if (u, t) in membership and membership[(u, t)] != c:
            print("Warning: (u,t) belongs to multiple communities:", (u, t), membership[(u, t)], c)
        membership[(u, t)] = c

# 初始化 P 结构
P = {}
for u in nodes:
    P[u] = {}

# 初始化边上出现过的所有 (u,t)，默认 0 向量
for u, v, t in time_links:
    for x in (u, v):
        if t not in P[x]:
            P[x][t] = torch.zeros(K, dtype=torch.float32)

# 再根据 community membership 覆盖对应 (u,t) 的向量为 one-hot
for (u, t), c in membership.items():
    if u not in P:
        P[u] = {}
    if t not in P[u]:
        P[u][t] = torch.zeros(K, dtype=torch.float32)
    vec = torch.zeros(K, dtype=torch.float32)
    vec[c] = 1.0
    P[u][t] = vec

edges = [(u, v, t) for (u, v, t) in time_links]



loss_cm, Q_CM = soft_l_modularity_loss(edges, P, omega=2, lex_type="ECM")
print("Q_CM =", Q_CM)
loss_jm, Q_JM = soft_l_modularity_loss(edges, P, omega=2, lex_type="EJM")
print("Q_JM =", Q_JM)
loss_mm, Q_MM = soft_l_modularity_loss(edges, P, omega=2, lex_type="EMM")
print("Q_MM =", Q_MM)

{0: tensor([ 0., 17.,  0.,  0.,  0.]), 1: tensor([ 0., 17.,  0.,  0.,  0.]), 2: tensor([11.,  6.,  0.,  0.,  0.]), 3: tensor([20.,  0.,  0.,  0.,  0.]), 4: tensor([15.,  0.,  0.,  0.,  0.])}
defaultdict(<class 'float'>, {2: 10.0, 3: 8.0, 0: 5.0, 1: 7.0, 4: 6.0})
20.0
tensor([20., 17.,  0.,  0.,  0.])
expectation:0.0
expectation:0.0
expectation:0.0
expectation:0.0
expectation:0.0
Modularity term: tensor([0.])
Q_CM = tensor([-0.1111])
{0: tensor([ 0., 17.,  0.,  0.,  0.]), 1: tensor([ 0., 17.,  0.,  0.,  0.]), 2: tensor([11.,  6.,  0.,  0.,  0.]), 3: tensor([20.,  0.,  0.,  0.,  0.]), 4: tensor([15.,  0.,  0.,  0.,  0.])}
defaultdict(<class 'float'>, {2: 10.0, 3: 8.0, 0: 5.0, 1: 7.0, 4: 6.0})
20.0
tensor([20., 17.,  0.,  0.,  0.])
expectation:0.02777777798473835
expectation:0.0
expectation:0.0
expectation:0.0
expectation:0.0
Modularity term: tensor([-0.0278])
Q_JM = tensor([-0.1389])
{0: tensor([ 0., 17.,  0.,  0.,  0.]), 1: tensor([ 0., 17.,  0.,  0.,  0.]), 2: tensor([11.,  6.,  0.,  0

In [99]:
import torch
from collections import defaultdict
from itertools import combinations_with_replacement


def compute_soft_Tu(P, K):
    """
    |T_{u∈C}| = Σ_t p(u,t,C)
    P[u][t] = 1D tensor(K)
    返回: Tu[u] -> 1D tensor(K)
    """
    device = next(iter(next(iter(P.values())).values())).device
    Tu = {}
    for u, td in P.items():
        acc = torch.zeros(K, device=device)
        for t, pvec in td.items():
            acc = acc + pvec
        Tu[u] = acc
    return Tu


def compute_soft_Tuv(P, K):
    """
    |T_{uv∈C}| = Σ_t p(u,t,C)p(v,t,C)
    包含 (u,u)！
    返回: Tuv[(u,v)] -> 1D tensor(K)
    """
    device = next(iter(next(iter(P.values())).values())).device
    Tuv = {}
    nodes = list(P.keys())

    for u, v in combinations_with_replacement(nodes, 2):
        acc = torch.zeros(K, device=device)

        # u 与 v 的共同时间
        t_common = set(P[u].keys()) & set(P[v].keys())
        
        for t in t_common:
            acc += P[u][t] * P[v][t]

        Tuv[(u, v)] = acc

    return Tuv


def compute_strength_C(P, K):
    """
    Strength_C(t) = 1 - ∏_u (1 - p(u,t,C))
    |T_C| = Σ_t Strength_C(t)

    返回:
        strength[c][t] = Strength_C(t) (float)
        T_C_len: 1D tensor(K) = 软社区存续时间
    """
    all_times = sorted({t for u in P for t in P[u].keys()})
    device = next(iter(next(iter(P.values())).values())).device

    strength = {c: {} for c in range(K)}
    for t in all_times:
        prod_term = torch.ones(K, device=device)
        for u, td in P.items():
            if t in td:
                prod_term = prod_term * (1 - td[t])
        strength_t = 1 - prod_term
        for c in range(K):
            strength[c][t] = float(strength_t[c].detach().cpu())

    T_C_len = torch.tensor(
        [sum(strength[c][t] for t in all_times) for c in range(K)],
        dtype=torch.float32,
        device=device,
    )
    return strength, T_C_len, all_times


def compute_soft_csc(P):
    """
    soft CSC:
    η_u = Σ_t 0.5 * ||p(u,t) - p(u,t_prev)||_1
    one-hot 时 ≈ 节点社区切换次数
    返回: eta_total (标量 tensor)
    """
    device = next(iter(next(iter(P.values())).values())).device
    eta_total = torch.zeros(1, device=device)
    for u, td in P.items():
        times = sorted(td.keys())
        for i in range(1, len(times)):
            diff = (td[times[i]] - td[times[i-1]]).abs().sum() * 0.5
            eta_total = eta_total + diff
    return eta_total.squeeze(0)


def soft_longitudinal_modularity_per_community(
    time_links,
    P,
    K,
    lex_type="MM",
    omega=2.0,
):
    """
    严格按照你给的硬分配代码结构来写的 soft 版本：

    - L_C: 每个社区的软内部连接数（对应 _get_communities_nb_interactions）
    - E_C: 每个社区的期望（对应 _get_communities_*mes）
    - Q = Σ_C [L_C/(2m) - E_C] - ω/(2m)*CSC

    参数：
        time_links: list[(u, v, t)]
        P[u][t]: 1D tensor(K)，soft 社区分配
        K: 社区数
        lex_type: "CM" / "JM" / "MM"
        omega: 时间平滑惩罚系数

    返回：
        Q: 标量 tensor （L-Modularity）
    """
    device = next(iter(next(iter(P.values())).values())).device
    # 所有节点 & 时间
    nodes = sorted(P.keys())
    all_times = sorted({t for (_, _, t) in time_links})
    T_len = float(max(all_times) - min(all_times) + 1)
    m = float(len(time_links))
    
    # 度数 k_u
    ku = defaultdict(float)
    for u, v, t in time_links:
        ku[u] += 1.0
        ku[v] += 1.0

    # 1. 每个社区的软内部“边数” L_C
    # 硬版：对每个 leaf 的 topo_neighbors，内部加 2**(... )
    # 这里：每条边 (u,v,t)，对每个 C 累加 2*p(u,t,C)p(v,t,C)
    L_C = torch.zeros(K, device=device)
    for u, v, t in time_links:
        pu = P[u][t]  # 1D tensor(K)
        pv = P[v][t]
        L_C = L_C + 2.0 * (pu * pv)

    # 2. 时间相关量：Tu, Tuv, T_C
    Tu = compute_soft_Tu(P, K)          # |T_{u∈C}|
    Tuv = compute_soft_Tuv(P, K)        # |T_{uv∈C}|
    strength, T_C_len, _ = compute_strength_C(P, K)  # |T_C|

    # 3. 每个社区的 expectation（软版 _get_communities_*mes）
    communities_expectations = torch.zeros(K, device=device)
    print(T_len)
    # 遍历社区 c，和节点对 (u,v)
    for c in range(K):
        print(T_C_len[c])
        community_nodes = [u for u in nodes if Tu[u][c] > 0]
        expectation_c = torch.zeros(1, device=device)
        for u, v in combinations_with_replacement(community_nodes, 2):
            factor = 2.0 if u != v else 1.0  # 对应 2**(source != target)
            k_uk_v = ku[u] * ku[v]
            print(f"u:{u}, v:{v}, factor: {factor}, k_uk_v:{k_uk_v}")
            if lex_type == "CM":  # ECM: co-membership
                Tuv_c = Tuv.get((u, v), torch.zeros(K, device=device))[c]
                print(f'u:{u}, v:{v}, c:{c}, Tuv_c:{Tuv_c}')
                time_factor = Tuv_c / T_len   # coexistence / network_duration

            elif lex_type == "JM":  # EJM: joint membership
                
                time_factor = T_C_len[c] / T_len  # community_duration / network_duration

            elif lex_type == "MM":  # EMM: mean membership（这里采用非 stream-graph 分支形式）
                Tu_c = Tu[u][c]
                Tv_c = Tu[v][c]
                geo_mean = torch.sqrt(Tu_c * Tv_c + 1e-12)
                time_factor = geo_mean / T_len   # geo_mean / network_duration (简化)

            else:
                raise ValueError('lex_type must be "CM", "JM", or "MM"')
            expected_value = factor * k_uk_v * time_factor / ((2.0 * m) ** 2)
            expectation_c = expectation_c + expected_value

        communities_expectations[c] = expectation_c

    # 4. 时间平滑惩罚项（soft CSC）
    eta_total = compute_soft_csc(P)
    time_penalty = -omega * eta_total / (2.0 * m)
    print(communities_expectations)
    # 5. Aggregation：Q = Σ_C L_C/(2m) - E_C + time_penalty
    Q = (L_C / (2.0 * m) - communities_expectations).sum() + time_penalty
    return Q




Q_CM = soft_longitudinal_modularity_per_community(edges, P, K=5, omega=0, lex_type="CM")
print("Q_CM =", Q_CM)
Q_JM = soft_longitudinal_modularity_per_community(edges, P,K=5, omega=0, lex_type="JM")
print("Q_JM =", Q_JM)
Q_MM = soft_longitudinal_modularity_per_community(edges, P, K=5,omega=0, lex_type="MM")
print("Q_MM =", Q_MM)

20.0
tensor(20.)
u:2, v:2, factor: 1.0, k_uk_v:100.0
u:2, v:2, c:0, Tuv_c:11.0
u:2, v:3, factor: 2.0, k_uk_v:80.0
u:2, v:3, c:0, Tuv_c:11.0
u:2, v:4, factor: 2.0, k_uk_v:60.0
u:2, v:4, c:0, Tuv_c:6.0
u:3, v:3, factor: 1.0, k_uk_v:64.0
u:3, v:3, c:0, Tuv_c:20.0
u:3, v:4, factor: 2.0, k_uk_v:48.0
u:3, v:4, c:0, Tuv_c:15.0
u:4, v:4, factor: 1.0, k_uk_v:36.0
u:4, v:4, c:0, Tuv_c:15.0
tensor(17.)
u:0, v:0, factor: 1.0, k_uk_v:25.0
u:0, v:0, c:1, Tuv_c:17.0
u:0, v:1, factor: 2.0, k_uk_v:35.0
u:0, v:1, c:1, Tuv_c:17.0
u:0, v:2, factor: 2.0, k_uk_v:50.0
u:0, v:2, c:1, Tuv_c:6.0
u:1, v:1, factor: 1.0, k_uk_v:49.0
u:1, v:1, c:1, Tuv_c:17.0
u:1, v:2, factor: 2.0, k_uk_v:70.0
u:1, v:2, c:1, Tuv_c:6.0
u:2, v:2, factor: 1.0, k_uk_v:100.0
u:2, v:2, c:1, Tuv_c:6.0
tensor(0.)
tensor(0.)
tensor(0.)
tensor([0.2639, 0.1731, 0.0000, 0.0000, 0.0000])
Q_CM = tensor(0.5630)
20.0
tensor(20.)
u:2, v:2, factor: 1.0, k_uk_v:100.0
u:2, v:3, factor: 2.0, k_uk_v:80.0
u:2, v:4, factor: 2.0, k_uk_v:60.0
u:3, v:3, fact

In [12]:
print(len(edges))

18


In [11]:
print(len(comm[1]))

46
